# LIAR Fake News Detection - Novel LIAR-Only Optimization v7

Binary-focused original LIAR run. This version fixes original/refit probability overwriting, adds a LIAR credibility calibrator into the ensemble, keeps the metadata GNN fusion, and writes fresh v7 result folders. No LIAR-PLUS evidence is used.


In [1]:
# Run this setup cell once if a package is missing in your Jupyter kernel.
%pip install pandas numpy scipy seaborn matplotlib spacy wordcloud nltk scikit-learn transformers datasets accelerate sentence-transformers torch tqdm sentencepiece
%pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl


Note: you may need to restart the kernel to use updated packages.
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - ------------------------------------- 0.5/12.8 MB 840.2 kB/s eta 0:00:15
     -- ------------------------------------- 0.8/12.8 MB 1.0 MB/s eta 0:00:12
     --- ------------------------------------ 1.0/12.8 MB 1.1 MB/s eta 0:00:11
     ---- ----------------------------------- 1.3/12.8 MB 1.2 MB/s eta 0:00:10
     ---- ----------------------------------- 1.3/12.8 MB 1.2 MB/s eta 0:00:10
     ---- ----------------------------------- 1.6/12.8 MB 1.0 MB/s eta 0:00:12
     ---- ----------------------------------- 1.6/12.8 MB 1.0 MB/s eta 0:00:12
     ----- --------------------------------- 1.8/12.8 MB 915.5 kB/s eta 0:00:12
     ------ -------------------------------- 2.1/12.8 MB 947.3 kB/s eta 0:00:12
    

In [2]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
import re
import json
import inspect
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import nltk
import spacy

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
    recall_score,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

nltk.download("stopwords")
nlp = spacy.load("en_core_web_sm")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


C:\Users\Asus\Downloads\Fake news detection\code\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Device: cuda
GPU: NVIDIA GeForce RTX 2060


In [3]:
PROJECT_DIR = Path(r"C:\Users\Asus\Downloads\Fake news detection\code")
DATA_DIR = PROJECT_DIR / "dataset"
RESULTS_DIR = PROJECT_DIR / "optimized_results_simple_liar_novel_v7"
LOGS_DIR = PROJECT_DIR / "optimized_logs_simple_liar_novel_v7"
SUMMARY_DIR = PROJECT_DIR / "optimized_summaries_simple_liar_novel_v7"

for path in [RESULTS_DIR, LOGS_DIR, SUMMARY_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RUN_VERSION = "v7"
print(f"Running notebook version: {RUN_VERSION}")
print(f"Results folder: {RESULTS_DIR}")

# Fresh retrain only. No older weight folders are pooled in v7.
EXTRA_LIAR_RESULT_DIRS = []

LIAR_COLUMNS = [
    "Id",
    "Label",
    "Statement",
    "Subject",
    "Speaker",
    "Job Title",
    "State",
    "Party",
    "Barely True Counts",
    "False Counts",
    "Half True Counts",
    "Mostly True Counts",
    "Pants on Fire Counts",
    "Context",
]


def load_liar_split(split_name):
    df = pd.read_csv(DATA_DIR / f"{split_name}.csv")
    df.columns = LIAR_COLUMNS
    df["split"] = split_name
    return df

train_raw = load_liar_split("train")
valid_raw = load_liar_split("valid")
test_raw = load_liar_split("test")

for split_name, df in [("train", train_raw), ("valid", valid_raw), ("test", test_raw)]:
    print(f"{split_name}: {df.shape}")
    print(df["Label"].value_counts().sort_index())
    print()


Running notebook version: v7
Results folder: C:\Users\Asus\Downloads\Fake news detection\code\optimized_results_simple_liar_novel_v7
train: (10269, 15)
Label
barely-true    1657
false          1998
half-true      2123
mostly-true    1966
pants-fire      842
true           1683
Name: count, dtype: int64

valid: (1284, 15)
Label
barely-true    237
false          263
half-true      248
mostly-true    251
pants-fire     116
true           169
Name: count, dtype: int64

test: (1283, 15)
Label
barely-true    214
false          250
half-true      267
mostly-true    249
pants-fire      92
true           211
Name: count, dtype: int64



In [4]:
# Paper-inspired LIAR input: combine statement + metadata.
# For transformer models, keep this light. Do not remove stopwords, punctuation, or numbers.
COUNT_COLUMNS = [
    "Barely True Counts",
    "False Counts",
    "Half True Counts",
    "Mostly True Counts",
    "Pants on Fire Counts",
]
TEXT_METADATA_COLUMNS = ["Subject", "Speaker", "Job Title", "State", "Party", "Context"]


def clean_field(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text in {"\\N", "nan", "None", "null"}:
        return ""
    return re.sub(r"\s+", " ", text)


def build_liar_text(row, include_history_counts=True):
    parts = []

    statement = clean_field(row["Statement"])
    if statement:
        parts.append(f"statement: {statement}")

    for col in TEXT_METADATA_COLUMNS:
        value = clean_field(row[col])
        if value:
            parts.append(f"{col.lower()}: {value}")

    if include_history_counts:
        counts = []
        count_names = {
            "Barely True Counts": "barely_true",
            "False Counts": "false",
            "Half True Counts": "half_true",
            "Mostly True Counts": "mostly_true",
            "Pants on Fire Counts": "pants_fire",
        }
        for col, name in count_names.items():
            value = row[col]
            if pd.notna(value):
                counts.append(f"{name}={int(value)}")
        if counts:
            parts.append("speaker_history: " + " ".join(counts))

    return " ".join(parts)


def add_model_text(df):
    df = df.copy()
    df["model_text"] = df.apply(build_liar_text, axis=1)
    df["text_length"] = df["model_text"].str.split().str.len()
    return df

train_df = add_model_text(train_raw)
valid_df = add_model_text(valid_raw)
test_df = add_model_text(test_raw)

print(train_df[["Label", "model_text", "text_length"]].head(3).to_string(index=False))
print("\nLength summary:")
print(pd.concat([train_df, valid_df, test_df])["text_length"].describe())


      Label                                                                                                                                                                                                                                                                                                                                                                                    model_text  text_length
      false                                                                                          statement: Says the Annies List political group supports third-trimester abortions on demand. subject: abortion speaker: dwayne-bohac job title: State representative state: Texas party: republican context: a mailer speaker_history: barely_true=0 false=1 half_true=0 mostly_true=0 pants_fire=0           33
  half-true statement: When did the decline of coal start? It started when natural gas took off that started to begin in (President George W.) Bushs administration. subject: energy,histo

In [5]:
# Binary mapping follows the paper: false/barely-true/pants-fire = fake, half/mostly/true = real.
BINARY_MAP = {
    "pants-fire": 0,
    "false": 0,
    "barely-true": 0,
    "half-true": 1,
    "mostly-true": 1,
    "true": 1,
}
BINARY_LABEL_NAMES = ["fake", "real"]

MULTICLASS_MAP = {
    "pants-fire": 0,
    "false": 1,
    "barely-true": 2,
    "half-true": 3,
    "mostly-true": 4,
    "true": 5,
}
MULTICLASS_LABEL_NAMES = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]

# Hierarchical multiclass setup for simple LIAR:
# binary fake/real gate, then a fake-subtype classifier and a real-subtype classifier.
FAKE_SUBTYPE_MAP = {
    "pants-fire": 0,
    "false": 1,
    "barely-true": 2,
}
FAKE_SUBTYPE_LABEL_NAMES = ["pants-fire", "false", "barely-true"]

REAL_SUBTYPE_MAP = {
    "half-true": 0,
    "mostly-true": 1,
    "true": 2,
}
REAL_SUBTYPE_LABEL_NAMES = ["half-true", "mostly-true", "true"]


def make_task_splits(label_map):
    splits = {}
    for split_name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
        task_df = df.copy()
        task_df["label"] = task_df["Label"].map(label_map)
        task_df = task_df.dropna(subset=["label", "model_text"])
        task_df["label"] = task_df["label"].astype(int)
        splits[split_name] = {
            "texts": task_df["model_text"].tolist(),
            "labels": task_df["label"].to_numpy(dtype=np.int64),
            "frame": task_df,
        }
    return splits


binary_splits = make_task_splits(BINARY_MAP)
multiclass_splits = make_task_splits(MULTICLASS_MAP)
fake_subtype_splits = make_task_splits(FAKE_SUBTYPE_MAP)
real_subtype_splits = make_task_splits(REAL_SUBTYPE_MAP)



def make_binary_nested_refit_splits(label_map, inner_valid_size=0.08):
    # Original LIAR only: use original valid as additional training data,
    # while holding out a small inner validation split from original train for early stopping/calibration.
    train_task = train_df.copy()
    train_task["label"] = train_task["Label"].map(label_map)
    train_task = train_task.dropna(subset=["label", "model_text"]).copy()
    train_task["label"] = train_task["label"].astype(int)

    original_valid_task = valid_df.copy()
    original_valid_task["label"] = original_valid_task["Label"].map(label_map)
    original_valid_task = original_valid_task.dropna(subset=["label", "model_text"]).copy()
    original_valid_task["label"] = original_valid_task["label"].astype(int)

    inner_train_df, inner_valid_df = train_test_split(
        train_task,
        test_size=inner_valid_size,
        stratify=train_task["label"],
        random_state=SEED,
    )
    final_train_df = pd.concat([inner_train_df, original_valid_task], ignore_index=True)

    test_task = test_df.copy()
    test_task["label"] = test_task["Label"].map(label_map)
    test_task = test_task.dropna(subset=["label", "model_text"]).copy()
    test_task["label"] = test_task["label"].astype(int)

    return {
        "train": to_split_payload(final_train_df),
        "valid": to_split_payload(inner_valid_df),
        "test": to_split_payload(test_task),
    }


def to_split_payload(task_df):
    return {
        "texts": task_df["model_text"].tolist(),
        "labels": task_df["label"].to_numpy(dtype=np.int64),
        "frame": task_df.reset_index(drop=True),
    }


binary_refit_splits = make_binary_nested_refit_splits(BINARY_MAP, inner_valid_size=0.08)

for task_name, splits, names in [
    ("binary", binary_splits, BINARY_LABEL_NAMES),
    ("multiclass", multiclass_splits, MULTICLASS_LABEL_NAMES),
    ("fake_subtype", fake_subtype_splits, FAKE_SUBTYPE_LABEL_NAMES),
    ("real_subtype", real_subtype_splits, REAL_SUBTYPE_LABEL_NAMES),
]:
    print(f"\n{task_name.upper()} distributions")
    for split_name in ["train", "valid", "test"]:
        labels = pd.Series(splits[split_name]["labels"]).map(dict(enumerate(names)))
        print(split_name)
        print(labels.value_counts().reindex(names, fill_value=0))



BINARY distributions
train
fake    4497
real    5772
Name: count, dtype: int64
valid
fake    616
real    668
Name: count, dtype: int64
test
fake    556
real    727
Name: count, dtype: int64

MULTICLASS distributions
train
pants-fire      842
false          1998
barely-true    1657
half-true      2123
mostly-true    1966
true           1683
Name: count, dtype: int64
valid
pants-fire     116
false          263
barely-true    237
half-true      248
mostly-true    251
true           169
Name: count, dtype: int64
test
pants-fire      92
false          250
barely-true    214
half-true      267
mostly-true    249
true           211
Name: count, dtype: int64

FAKE_SUBTYPE distributions
train
pants-fire      842
false          1998
barely-true    1657
Name: count, dtype: int64
valid
pants-fire     116
false          263
barely-true    237
Name: count, dtype: int64
test
pants-fire      92
false          250
barely-true    214
Name: count, dtype: int64

REAL_SUBTYPE distributions
train
half-true

In [6]:
def make_compute_metrics(task_name, label_names):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        metrics = {
            "accuracy": accuracy_score(labels, preds),
            "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
            "weighted_recall": recall_score(labels, preds, average="weighted", zero_division=0),
            "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
            "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        }
        if task_name == "binary":
            metrics["fake_recall"] = recall_score(labels, preds, pos_label=0, zero_division=0)
            metrics["real_recall"] = recall_score(labels, preds, pos_label=1, zero_division=0)
        return metrics
    return compute_metrics


def summarize_predictions(y_true, y_pred, label_names, task_name):
    summary = make_compute_metrics(task_name, label_names)((None, y_true)) if False else {}
    summary.update({
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    })
    if task_name == "binary":
        summary["fake_recall"] = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
        summary["real_recall"] = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    return summary


def print_report(title, y_true, y_pred, label_names, task_name):
    print("\n" + title)
    summary = summarize_predictions(y_true, y_pred, label_names, task_name)
    print(pd.Series(summary).round(4).to_string())
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
    return summary


In [7]:
class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = [int(x) for x in labels]

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, label_smoothing=0.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.label_smoothing = float(label_smoothing)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights, label_smoothing=self.label_smoothing)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def encode_texts(tokenizer, texts, labels, max_length=128):
    encodings = tokenizer(texts, truncation=True, max_length=max_length)
    return FakeNewsDataset(encodings, labels)


def model_safe_name(model_name):
    return model_name.replace("/", "_").replace("-", "_").replace(".", "_")


def local_model_ready(path):
    path = Path(path)
    model_file_exists = (path / "model.safetensors").exists() or (path / "pytorch_model.bin").exists()
    tokenizer_exists = any((path / name).exists() for name in ["tokenizer.json", "vocab.txt", "merges.txt", "sentencepiece.bpe.model", "spiece.model", "spm.model"])
    return (path / "config.json").exists() and model_file_exists and tokenizer_exists


def get_class_weights(labels, num_labels):
    classes = np.arange(num_labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=np.asarray(labels))
    return torch.tensor(weights, dtype=torch.float32)


def cleanup_checkpoints(run_dir):
    run_dir = Path(run_dir)
    for checkpoint_dir in run_dir.glob("checkpoint-*"):
        if checkpoint_dir.is_dir():
            shutil.rmtree(checkpoint_dir, ignore_errors=True)


# Conservative simple-LIAR tuning: safe default models, early stopping,
# validation-selected checkpointing, and optional slow candidates kept out of the automatic run.

MODEL_CONFIGS = {
    "distilbert-base-uncased": {"learning_rate": 3e-5, "epochs": 2, "batch_size": 16, "grad_accum": 1, "max_length": 128, "eval_batch_size": 32, "use_safetensors": True, "label_smoothing": 0.0, "early_stop": 1},
    "bert-base-uncased": {"learning_rate": 3e-5, "epochs": 2, "batch_size": 16, "grad_accum": 1, "max_length": 128, "eval_batch_size": 32, "use_safetensors": True, "label_smoothing": 0.0, "early_stop": 1},
    "roberta-base": {"learning_rate": 2e-5, "epochs": 2, "batch_size": 16, "grad_accum": 1, "max_length": 128, "eval_batch_size": 32, "use_safetensors": True, "label_smoothing": 0.0, "early_stop": 1},
    "distilroberta-base": {"learning_rate": 2e-5, "epochs": 3, "batch_size": 16, "grad_accum": 1, "max_length": 128, "eval_batch_size": 32, "use_safetensors": True, "label_smoothing": 0.0, "early_stop": 1},
    "albert-base-v2": {"learning_rate": 2e-5, "epochs": 3, "batch_size": 16, "grad_accum": 1, "max_length": 128, "eval_batch_size": 32, "use_safetensors": True, "label_smoothing": 0.0, "early_stop": 1},
    "albert-large-v2": {"learning_rate": 1e-5, "epochs": 3, "batch_size": 4, "grad_accum": 4, "max_length": 128, "eval_batch_size": 8, "use_safetensors": True, "label_smoothing": 0.0, "early_stop": 1},
    "roberta-large": {"learning_rate": 7e-6, "epochs": 2, "batch_size": 2, "grad_accum": 16, "max_length": 128, "eval_batch_size": 4, "use_safetensors": True, "label_smoothing": 0.0, "early_stop": 1},
}

PAPER_MODELS = ["distilbert-base-uncased", "bert-base-uncased", "roberta-base"]
ADDITIONAL_LIAR_TRANSFORMERS = []
DEFAULT_MODELS = PAPER_MODELS + ADDITIONAL_LIAR_TRANSFORMERS
BINARY_HIGH_ACCURACY_MODELS = DEFAULT_MODELS
BINARY_REFIT_MODELS = ["distilbert-base-uncased", "bert-base-uncased", "roberta-base"]
OPTIONAL_SLOW_MODELS = ["distilroberta-base", "albert-base-v2", "albert-large-v2", "roberta-large"]
HIERARCHICAL_MODELS = ["distilbert-base-uncased"]


In [8]:
def torch_version_tuple():
    base = torch.__version__.split("+")[0]
    parts = []
    for piece in base.split(".")[:3]:
        try:
            parts.append(int(piece))
        except ValueError:
            parts.append(0)
    while len(parts) < 3:
        parts.append(0)
    return tuple(parts)


def torch_can_load_bin_weights():
    return torch_version_tuple() >= (2, 6, 0)


def estimate_warmup_steps(num_examples, batch_size, grad_accum, epochs):
    effective_batch = max(1, int(batch_size) * int(grad_accum))
    steps_per_epoch = max(1, int(np.ceil(num_examples / effective_batch)))
    total_steps = max(1, steps_per_epoch * int(np.ceil(epochs)))
    return min(150, max(10, int(0.08 * total_steps)))


def make_training_arguments(**kwargs):
    supported = set(inspect.signature(TrainingArguments.__init__).parameters)
    args = dict(kwargs)
    if "eval_strategy" in args and "eval_strategy" not in supported and "evaluation_strategy" in supported:
        args["evaluation_strategy"] = args.pop("eval_strategy")
    filtered = {key: value for key, value in args.items() if key in supported}
    skipped = sorted(set(args) - supported)
    if skipped:
        print("Skipping unsupported TrainingArguments:", skipped)
    return TrainingArguments(**filtered)


def build_weighted_trainer(tokenizer, **kwargs):
    supported = set(inspect.signature(Trainer.__init__).parameters)
    if "processing_class" in supported:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in supported:
        kwargs["tokenizer"] = tokenizer
    return WeightedLossTrainer(**kwargs)


def train_or_load_transformer(
    model_name,
    splits,
    task_name,
    label_names,
    force_retrain=False,
    use_class_weights=True,
):
    config = MODEL_CONFIGS[model_name]
    num_labels = len(label_names)
    run_dir = RESULTS_DIR / f"{model_safe_name(model_name)}_{task_name}"
    if force_retrain and run_dir.exists():
        shutil.rmtree(run_dir, ignore_errors=True)
    run_dir.mkdir(parents=True, exist_ok=True)

    source = str(run_dir) if local_model_ready(run_dir) and not force_retrain else model_name
    if source == str(run_dir):
        print(f"Loading existing optimized model: {run_dir}")
    else:
        print(f"Training {model_name} for {task_name} -> {run_dir}")

    if config.get("requires_torch_26") and not torch_can_load_bin_weights():
        print(f"Skipping {model_name}: this model requires torch>=2.6 in the current transformers setup.")
        return {"model_name": model_name, "task_name": task_name, "run_dir": run_dir, "skipped": True}

    tokenizer = AutoTokenizer.from_pretrained(source)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    train_dataset = encode_texts(tokenizer, splits["train"]["texts"], splits["train"]["labels"], config["max_length"])
    valid_dataset = encode_texts(tokenizer, splits["valid"]["texts"], splits["valid"]["labels"], config["max_length"])
    test_dataset = encode_texts(tokenizer, splits["test"]["texts"], splits["test"]["labels"], config["max_length"])

    id2label = {i: name for i, name in enumerate(label_names)}
    label2id = {name: i for i, name in id2label.items()}
    model_kwargs = {
        "num_labels": num_labels,
        "id2label": id2label,
        "label2id": label2id,
        "ignore_mismatched_sizes": True,
    }
    if config.get("use_safetensors"):
        model_kwargs["use_safetensors"] = True
    model = AutoModelForSequenceClassification.from_pretrained(source, **model_kwargs)

    warmup_steps = estimate_warmup_steps(
        len(splits["train"]["labels"]),
        config["batch_size"],
        config.get("grad_accum", 1),
        config["epochs"],
    )
    training_args = make_training_arguments(
        output_dir=str(run_dir),
        logging_dir=str(LOGS_DIR / f"{model_safe_name(model_name)}_{task_name}"),
        do_train=True,
        do_eval=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model=config.get("metric_for_best_model", "eval_macro_f1"),
        greater_is_better=True,
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config.get("eval_batch_size", 32),
        gradient_accumulation_steps=config.get("grad_accum", 1),
        learning_rate=config["learning_rate"],
        num_train_epochs=config["epochs"],
        weight_decay=config.get("weight_decay", 0.02),
        warmup_steps=warmup_steps,
        logging_steps=50,
        save_total_limit=1,
        save_only_model=True,
        seed=SEED,
        data_seed=SEED,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    class_weights = get_class_weights(splits["train"]["labels"], num_labels) if use_class_weights else None
    trainer = build_weighted_trainer(
        tokenizer,
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        data_collator=data_collator,
        compute_metrics=make_compute_metrics(task_name, label_names),
        class_weights=class_weights,
        label_smoothing=config.get("label_smoothing", 0.0),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config.get("early_stop", 2))],
    )

    if source != str(run_dir):
        trainer.train()
        trainer.save_model(str(run_dir))
        tokenizer.save_pretrained(str(run_dir))
        cleanup_checkpoints(run_dir)

    valid_metrics = trainer.evaluate(valid_dataset)
    test_metrics = trainer.evaluate(test_dataset)
    print(f"\n{model_name} {task_name} validation metrics")
    print(pd.Series(valid_metrics).filter(like="eval_").round(4).to_string())
    print(f"\n{model_name} {task_name} test metrics")
    print(pd.Series(test_metrics).filter(like="eval_").round(4).to_string())

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "model_name": model_name,
        "task_name": task_name,
        "run_dir": run_dir,
        "valid_metrics": valid_metrics,
        "test_metrics": test_metrics,
    }


In [9]:
# Full LIAR optimization run. This retrains into optimized_results_simple_liar_novel_v7 and leaves earlier results untouched.
# After one successful full run, set FORCE_RETRAIN=False and RUN_*_TRAINING=False to only re-evaluate.
FORCE_RETRAIN = False
RUN_BINARY_TRAINING = True
BINARY_MODELS = globals().get("BINARY_HIGH_ACCURACY_MODELS", DEFAULT_MODELS)

binary_artifacts = []
if RUN_BINARY_TRAINING:
    for model_name in BINARY_MODELS:
        artifact = train_or_load_transformer(
            model_name=model_name,
            splits=binary_splits,
            task_name="binary",
            label_names=BINARY_LABEL_NAMES,
            force_retrain=FORCE_RETRAIN,
            use_class_weights=True,
        )
        if not artifact.get("skipped"):
            binary_artifacts.append(artifact)

binary_artifacts


# Novel LIAR-only nested-refit models for binary accuracy: train with original valid added to training,
# and use a small inner validation split from original train for early stopping.
RUN_BINARY_REFIT_TRAINING = True
BINARY_REFIT_MODELS = globals().get("BINARY_REFIT_MODELS", ["distilbert-base-uncased", "bert-base-uncased", "roberta-base"])

binary_refit_artifacts = []
if RUN_BINARY_REFIT_TRAINING:
    for model_name in BINARY_REFIT_MODELS:
        artifact = train_or_load_transformer(
            model_name=model_name,
            splits=binary_refit_splits,
            task_name="binary_refit",
            label_names=BINARY_LABEL_NAMES,
            force_retrain=FORCE_RETRAIN,
            use_class_weights=True,
        )
        if not artifact.get("skipped"):
            binary_refit_artifacts.append(artifact)

binary_refit_artifacts


Training distilbert-base-uncased for binary -> C:\Users\Asus\Downloads\Fake news detection\code\optimized_results_simple_liar_novel_v7\distilbert_base_uncased_binary


Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 5882.12it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOG

Epoch,Training Loss,Validation Loss,Accuracy,Macro Recall,Weighted Recall,Macro F1,Weighted F1,Fake Recall,Real Recall
1,0.527724,0.556991,0.718069,0.708823,0.718069,0.698129,0.701271,0.480519,0.937126
2,0.478627,0.483337,0.726636,0.725460,0.726636,0.725700,0.726349,0.696429,0.754491


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.25it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Macro Recall,Weighted Recall,Macro F1,Weighted F1,Fake Recall,Real Recall
0.478627,0.483337,2,0.726636,0.725460,0.726636,0.725700,0.726349,0.696429,0.754491


...

Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.67it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Macro Recall,Weighted Recall,Macro F1,Weighted F1
0.458773,0.479082,2,0.763990,0.757540,0.763990,0.758851,0.763219


Training Loss,Validation Loss,Epoch,Accuracy,Macro Recall,Weighted Recall,Macro F1,Weighted F1
0.458773,0.506376,2,0.753702,0.746921,0.753702,0.748000,0.753052



roberta-base binary_refit validation metrics
eval_loss               0.4791
eval_accuracy           0.7640
eval_macro_recall       0.7575
eval_weighted_recall    0.7640
eval_macro_f1           0.7589
eval_weighted_f1        0.7632

roberta-base binary_refit test metrics
eval_loss               0.5064
eval_accuracy           0.7537
eval_macro_recall       0.7469
eval_weighted_recall    0.7537
eval_macro_f1           0.7480
eval_weighted_f1        0.7531


[{'model_name': 'distilbert-base-uncased',
  'task_name': 'binary_refit',
  'run_dir': WindowsPath('C:/Users/Asus/Downloads/Fake news detection/code/optimized_results_simple_liar_novel_v7/distilbert_base_uncased_binary_refit'),
  'valid_metrics': {'eval_loss': 0.47715869545936584,
   'eval_accuracy': 0.7591240875912408,
   'eval_macro_recall': 0.7559704184704185,
   'eval_weighted_recall': 0.7591240875912408,
   'eval_macro_f1': 0.7556509728561134,
   'eval_weighted_f1': 0.7592658473763481},
  'test_metrics': {'eval_loss': 0.4995276927947998,
   'eval_accuracy': 0.7521434138737334,
   'eval_macro_recall': 0.7476608809238716,
   'eval_weighted_recall': 0.7521434138737334,
   'eval_macro_f1': 0.7476608809238716,
   'eval_weighted_f1': 0.7521434138737334}},
 {'model_name': 'bert-base-uncased',
  'task_name': 'binary_refit',
  'run_dir': WindowsPath('C:/Users/Asus/Downloads/Fake news detection/code/optimized_results_simple_liar_novel_v7/bert_base_uncased_binary_refit'),
  'valid_metrics': 

In [10]:
def predict_probabilities(model_dir, texts, num_labels, max_length=128, batch_size=32):
    model_dir = Path(model_dir)
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir), num_labels=num_labels, local_files_only=True).to(device)
    model.eval()

    if not torch.cuda.is_available():
        batch_size = min(batch_size, 8)

    all_probs = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(device)
        with torch.inference_mode():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        all_probs.append(probs)

    result = np.vstack(all_probs)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


def get_ready_artifacts(task_name, model_names):
    artifacts = []
    for model_name in model_names:
        run_dir = RESULTS_DIR / f"{model_safe_name(model_name)}_{task_name}"
        if local_model_ready(run_dir):
            artifacts.append({"model_name": model_name, "task_name": task_name, "run_dir": run_dir})
        else:
            print(f"Skipping {model_name}; no trained model found at {run_dir}")
    return artifacts


def artifact_probability_key(artifact):
    task_name = artifact.get("task_name", "model")
    return f"{artifact['model_name']} [{task_name}]"


def collect_probabilities(artifacts, splits, num_labels, max_length=128):
    valid_probs = {}
    test_probs = {}
    for artifact in artifacts:
        model_name = artifact["model_name"]
        name = artifact_probability_key(artifact)
        cfg = MODEL_CONFIGS.get(model_name, {})
        model_max_length = cfg.get("max_length", max_length)
        model_batch_size = cfg.get("eval_batch_size", 32)
        print(f"Predicting probabilities: {name} (max_length={model_max_length}, batch_size={model_batch_size})")
        valid_probs[name] = predict_probabilities(artifact["run_dir"], splits["valid"]["texts"], num_labels, max_length=model_max_length, batch_size=model_batch_size)
        test_probs[name] = predict_probabilities(artifact["run_dir"], splits["test"]["texts"], num_labels, max_length=model_max_length, batch_size=model_batch_size)
    return valid_probs, test_probs


def collect_probabilities_for_texts(artifacts, texts, num_labels, split_name="all", max_length=128):
    prob_dict = {}
    for artifact in artifacts:
        model_name = artifact["model_name"]
        name = artifact_probability_key(artifact)
        cfg = MODEL_CONFIGS.get(model_name, {})
        model_max_length = cfg.get("max_length", max_length)
        model_batch_size = cfg.get("eval_batch_size", 32)
        print(f"Predicting {split_name} probabilities: {name} (max_length={model_max_length}, batch_size={model_batch_size})")
        prob_dict[name] = predict_probabilities(
            artifact["run_dir"],
            texts,
            num_labels,
            max_length=model_max_length,
            batch_size=model_batch_size,
        )
    return prob_dict


def average_probabilities(prob_dict, model_names=None, weights=None):
    if not prob_dict:
        return None
    if model_names is None:
        model_names = sorted(prob_dict)
    arrays = [prob_dict[name] for name in model_names]
    if weights is None:
        return np.mean(arrays, axis=0)
    weight_array = np.asarray([weights.get(name, 1.0) for name in model_names], dtype=float)
    if not np.isfinite(weight_array).all() or weight_array.sum() <= 0:
        weight_array = np.ones(len(model_names), dtype=float)
    weight_array = weight_array / weight_array.sum()
    return np.tensordot(weight_array, np.stack(arrays, axis=0), axes=(0, 0))


def majority_vote(prob_dict):
    model_names = sorted(prob_dict)
    model_preds = np.vstack([prob_dict[name].argmax(axis=1) for name in model_names]).T
    avg_probs = average_probabilities(prob_dict, model_names)
    final = []
    for row_idx, row in enumerate(model_preds):
        values, counts = np.unique(row, return_counts=True)
        winners = values[counts == counts.max()]
        if len(winners) == 1:
            final.append(int(winners[0]))
        else:
            final.append(int(winners[np.argmax(avg_probs[row_idx, winners])]))
    return np.asarray(final)


def summed_probability_vote(prob_dict):
    avg_probs = average_probabilities(prob_dict)
    return avg_probs.argmax(axis=1), avg_probs


def subset_probability_vote(prob_dict, model_names):
    avg_probs = average_probabilities(prob_dict, list(model_names))
    return avg_probs.argmax(axis=1), avg_probs


def validation_weighted_probability_vote(valid_prob_dict, test_prob_dict, y_valid, task_name):
    weights = {}
    for name in sorted(valid_prob_dict):
        valid_preds = valid_prob_dict[name].argmax(axis=1)
        weights[name] = f1_score(y_valid, valid_preds, average="macro", zero_division=0)
    test_avg = average_probabilities(test_prob_dict, sorted(test_prob_dict), weights)
    return test_avg.argmax(axis=1), test_avg, weights


def select_best_subset_ensemble(valid_prob_dict, test_prob_dict, y_valid, label_names, task_name, primary_metric="macro_f1"):
    import itertools
    rows = []
    names = sorted(valid_prob_dict)
    for size in range(1, len(names) + 1):
        for subset in itertools.combinations(names, size):
            valid_preds, _ = subset_probability_vote(valid_prob_dict, subset)
            summary = summarize_predictions(y_valid, valid_preds, label_names, task_name)
            rows.append({
                "models": subset,
                "model_count": size,
                **summary,
            })
    table = pd.DataFrame(rows)
    best = table.sort_values([primary_metric, "accuracy", "macro_recall"], ascending=False).iloc[0]
    best_models = tuple(best["models"])
    test_preds, test_probs = subset_probability_vote(test_prob_dict, best_models)
    return best_models, test_preds, test_probs, table


def evaluate_ensembles(valid_prob_dict, test_prob_dict, y_valid, y_test, label_names, task_name, split_name="test"):
    results = {}
    if not test_prob_dict:
        print(f"No probabilities available for {task_name} {split_name}.")
        return results

    majority_preds = majority_vote(test_prob_dict)
    results["majority_vote"] = {
        "preds": majority_preds,
        "summary": print_report(f"{task_name} {split_name} - Majority vote", y_test, majority_preds, label_names, task_name),
    }

    summed_preds, averaged_probs = summed_probability_vote(test_prob_dict)
    results["highest_sum"] = {
        "preds": summed_preds,
        "probs": averaged_probs,
        "summary": print_report(f"{task_name} {split_name} - Highest-sum / averaged probabilities", y_test, summed_preds, label_names, task_name),
    }

    weighted_preds, weighted_probs, model_weights = validation_weighted_probability_vote(valid_prob_dict, test_prob_dict, y_valid, task_name)
    results["validation_weighted"] = {
        "preds": weighted_preds,
        "probs": weighted_probs,
        "weights": model_weights,
        "summary": print_report(f"{task_name} {split_name} - Validation-weighted probability ensemble", y_test, weighted_preds, label_names, task_name),
    }

    selected_models, selected_preds, selected_probs, subset_table = select_best_subset_ensemble(
        valid_prob_dict,
        test_prob_dict,
        y_valid,
        label_names,
        task_name,
        primary_metric="macro_f1",
    )
    results["validation_selected_avg"] = {
        "models": selected_models,
        "preds": selected_preds,
        "probs": selected_probs,
        "subset_table": subset_table,
        "summary": print_report(
            f"{task_name} {split_name} - Validation-selected averaged probabilities {selected_models}",
            y_test,
            selected_preds,
            label_names,
            task_name,
        ),
    }

    return results


def tune_binary_threshold(fake_probs, y_true, optimize_for="accuracy", min_fake_recall=None):
    rows = []
    for threshold in np.linspace(0.05, 0.95, 181):
        preds = np.where(fake_probs >= threshold, 0, 1)
        rows.append({
            "threshold": float(threshold),
            "accuracy": accuracy_score(y_true, preds),
            "fake_recall": recall_score(y_true, preds, pos_label=0, zero_division=0),
            "real_recall": recall_score(y_true, preds, pos_label=1, zero_division=0),
            "macro_recall": recall_score(y_true, preds, average="macro", zero_division=0),
            "macro_f1": f1_score(y_true, preds, average="macro", zero_division=0),
        })
    table = pd.DataFrame(rows)
    candidates = table
    if min_fake_recall is not None:
        eligible = table[table["fake_recall"] >= min_fake_recall]
        if len(eligible):
            candidates = eligible
    best = candidates.sort_values([optimize_for, "macro_f1", "macro_recall", "accuracy"], ascending=False).iloc[0]
    return best, table


def numeric_feature_matrix(frame):
    count_cols = [
        "Barely True Counts",
        "False Counts",
        "Half True Counts",
        "Mostly True Counts",
        "Pants on Fire Counts",
    ]
    counts = frame[count_cols].fillna(0).astype(float).to_numpy()
    total = counts.sum(axis=1, keepdims=True)
    ratios = counts / np.maximum(total, 1.0)
    fake_ratio = counts[:, [0, 1, 4]].sum(axis=1, keepdims=True) / np.maximum(total, 1.0)
    real_ratio = counts[:, [2, 3]].sum(axis=1, keepdims=True) / np.maximum(total, 1.0)
    credibility_margin = real_ratio - fake_ratio
    text_len = frame["text_length"].fillna(0).astype(float).to_numpy().reshape(-1, 1)
    subject_count = frame["Subject"].fillna("").astype(str).str.split(",").str.len().to_numpy(dtype=float).reshape(-1, 1)
    return np.hstack([counts, ratios, total, fake_ratio, real_ratio, credibility_margin, text_len, subject_count])


def probability_features(prob_dict):
    if not prob_dict:
        return None
    ordered = [prob_dict[name] for name in sorted(prob_dict)]
    _, avg_probs = summed_probability_vote(prob_dict)
    return np.hstack(ordered + [avg_probs])


CREDIBILITY_COLUMNS = ["Speaker", "Subject", "Party", "State", "Context", "Job Title"]


def fit_smoothed_target_encoding(frame, labels, num_labels, columns=CREDIBILITY_COLUMNS, alpha=50.0):
    labels = np.asarray(labels, dtype=int)
    prior = np.bincount(labels, minlength=num_labels).astype(float)
    prior = prior / np.maximum(prior.sum(), 1.0)
    encoders = {}
    for column in columns:
        keys = frame[column].fillna("").astype(str).map(clean_field)
        table = {}
        for key, label in zip(keys, labels):
            bucket = table.setdefault(key, np.zeros(num_labels, dtype=float))
            bucket[label] += 1.0
        encoders[column] = {
            key: (counts + alpha * prior) / (counts.sum() + alpha)
            for key, counts in table.items()
        }
    return {"prior": prior, "encoders": encoders, "columns": list(columns)}


def transform_smoothed_target_encoding(frame, fitted):
    prior = fitted["prior"]
    parts = []
    for column in fitted["columns"]:
        mapping = fitted["encoders"][column]
        keys = frame[column].fillna("").astype(str).map(clean_field)
        encoded = np.vstack([mapping.get(key, prior) for key in keys])
        parts.append(encoded)
    return np.hstack(parts)


def credibility_feature_matrix(frame, fitted=None):
    parts = [numeric_feature_matrix(frame)]
    if fitted is not None:
        parts.append(transform_smoothed_target_encoding(frame, fitted))
    return np.hstack(parts)


def credibility_probability_baseline(splits, label_names, task_name):
    num_labels = len(label_names)
    fitted = fit_smoothed_target_encoding(
        splits["train"]["frame"],
        splits["train"]["labels"],
        num_labels,
    )
    x_train = credibility_feature_matrix(splits["train"]["frame"], fitted)
    x_valid = credibility_feature_matrix(splits["valid"]["frame"], fitted)
    x_test = credibility_feature_matrix(splits["test"]["frame"], fitted)

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_valid = scaler.transform(x_valid)
    x_test = scaler.transform(x_test)

    clf = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        C=2.0,
        solver="lbfgs",
        random_state=SEED,
    )
    clf.fit(x_train, splits["train"]["labels"])
    valid_probs = clf.predict_proba(x_valid)
    test_probs = clf.predict_proba(x_test)

    print_report(
        f"{task_name} valid - LIAR credibility baseline",
        splits["valid"]["labels"],
        valid_probs.argmax(axis=1),
        label_names,
        task_name,
    )
    test_summary = print_report(
        f"{task_name} test - LIAR credibility baseline",
        splits["test"]["labels"],
        test_probs.argmax(axis=1),
        label_names,
        task_name,
    )
    return valid_probs, test_probs, test_summary


def tfidf_probability_baseline(splits, label_names, task_name):
    tfidf = FeatureUnion([
        ("word", TfidfVectorizer(max_features=90000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("char", TfidfVectorizer(max_features=40000, analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
    ])
    clf = LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=2.0,
        solver="saga",
        n_jobs=-1,
        random_state=SEED,
    )
    x_train = tfidf.fit_transform(splits["train"]["texts"])
    clf.fit(x_train, splits["train"]["labels"])

    valid_probs = clf.predict_proba(tfidf.transform(splits["valid"]["texts"]))
    test_probs = clf.predict_proba(tfidf.transform(splits["test"]["texts"]))

    print_report(
        f"{task_name} valid - TF-IDF LogisticRegression baseline",
        splits["valid"]["labels"],
        valid_probs.argmax(axis=1),
        label_names,
        task_name,
    )
    test_summary = print_report(
        f"{task_name} test - TF-IDF LogisticRegression baseline",
        splits["test"]["labels"],
        test_probs.argmax(axis=1),
        label_names,
        task_name,
    )
    return valid_probs, test_probs, test_summary


def stacked_meta_ensemble(task_name, splits, valid_prob_dict, test_prob_dict, label_names):
    tfidf_valid_probs, tfidf_test_probs, tfidf_summary = tfidf_probability_baseline(splits, label_names, task_name)
    credibility_valid_probs, credibility_test_probs, credibility_summary = credibility_probability_baseline(splits, label_names, task_name)

    valid_transformer_features = probability_features(valid_prob_dict)
    test_transformer_features = probability_features(test_prob_dict)
    valid_parts = [tfidf_valid_probs, credibility_valid_probs]
    test_parts = [tfidf_test_probs, credibility_test_probs]

    if valid_transformer_features is not None:
        valid_parts.insert(0, valid_transformer_features)
        test_parts.insert(0, test_transformer_features)

    scaler = StandardScaler()
    scaler.fit(numeric_feature_matrix(splits["train"]["frame"]))
    valid_parts.append(scaler.transform(numeric_feature_matrix(splits["valid"]["frame"])))
    test_parts.append(scaler.transform(numeric_feature_matrix(splits["test"]["frame"])))

    x_meta_valid = np.hstack(valid_parts)
    x_meta_test = np.hstack(test_parts)
    y_valid = splits["valid"]["labels"]
    y_test = splits["test"]["labels"]

    meta = LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=1.0,
        solver="lbfgs",
        random_state=SEED,
    )
    meta.fit(x_meta_valid, y_valid)
    meta_test_probs = meta.predict_proba(x_meta_test)
    meta_test_preds = meta_test_probs.argmax(axis=1)
    meta_summary = print_report(
        f"{task_name} test - Stacked meta ensemble",
        y_test,
        meta_test_preds,
        label_names,
        task_name,
    )

    return {
        "tfidf_summary": tfidf_summary,
        "tfidf_test_probs": tfidf_test_probs,
        "tfidf_test_preds": tfidf_test_probs.argmax(axis=1),
        "credibility_summary": credibility_summary,
        "credibility_test_probs": credibility_test_probs,
        "credibility_test_preds": credibility_test_probs.argmax(axis=1),
        "meta_summary": meta_summary,
        "meta_test_probs": meta_test_probs,
        "meta_test_preds": meta_test_preds,
    }


# ===== LIAR-only metadata graph neural network =====
GRAPH_METADATA_COLUMNS = ["Speaker", "Subject", "Party", "State", "Context", "Job Title"]


def metadata_tokens(value, column_name):
    text = clean_field(value).lower()
    if not text:
        return [f"{column_name}:__missing__"]
    if column_name == "Subject":
        pieces = re.split(r"[,;/|]", text)
    else:
        pieces = [text]
    tokens = [piece.strip() for piece in pieces if piece.strip()]
    if not tokens:
        tokens = ["__missing__"]
    return [f"{column_name}:{token}" for token in tokens]


class SimpleGCN(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.25):
        super().__init__()
        self.fc1 = torch.nn.Linear(in_dim, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, out_dim)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, features, adjacency):
        x = torch.sparse.mm(adjacency, features)
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = torch.sparse.mm(adjacency, x)
        return self.fc2(x)


def build_liar_metadata_graph(splits, num_labels):
    split_order = ["train", "valid", "test"]
    frames = [splits[name]["frame"].reset_index(drop=True) for name in split_order]
    labels = {name: splits[name]["labels"] for name in split_order}
    claim_offsets = {}
    node_count = 0
    for name, frame in zip(split_order, frames):
        claim_offsets[name] = node_count
        node_count += len(frame)

    token_to_id = {}
    edges = []
    token_type = {}
    for split_name, frame in zip(split_order, frames):
        offset = claim_offsets[split_name]
        for row_idx, row in frame.iterrows():
            claim_node = offset + row_idx
            for col_idx, column in enumerate(GRAPH_METADATA_COLUMNS):
                for token in metadata_tokens(row.get(column, ""), column):
                    if token not in token_to_id:
                        token_to_id[token] = node_count
                        token_type[token] = col_idx
                        node_count += 1
                    meta_node = token_to_id[token]
                    edges.append((claim_node, meta_node))
                    edges.append((meta_node, claim_node))

    # Self-loops.
    edges.extend((idx, idx) for idx in range(node_count))
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    values = torch.ones(edge_index.shape[1], dtype=torch.float32)
    degree = torch.zeros(node_count, dtype=torch.float32)
    degree.scatter_add_(0, edge_index[0], values)
    norm_values = values / torch.sqrt(degree[edge_index[0]].clamp_min(1.0) * degree[edge_index[1]].clamp_min(1.0))
    adjacency = torch.sparse_coo_tensor(edge_index, norm_values, (node_count, node_count)).coalesce()

    numeric_scaler = StandardScaler()
    numeric_scaler.fit(numeric_feature_matrix(splits["train"]["frame"]))
    numeric_dim = numeric_feature_matrix(splits["train"]["frame"]).shape[1]
    type_dim = 1 + len(GRAPH_METADATA_COLUMNS)
    features = np.zeros((node_count, numeric_dim + type_dim), dtype=np.float32)

    for split_name, frame in zip(split_order, frames):
        offset = claim_offsets[split_name]
        numeric = numeric_scaler.transform(numeric_feature_matrix(frame)).astype(np.float32)
        features[offset:offset + len(frame), :numeric_dim] = numeric
        features[offset:offset + len(frame), numeric_dim] = 1.0  # claim node type

    for token, node_id in token_to_id.items():
        features[node_id, numeric_dim + 1 + token_type[token]] = 1.0

    idx = {
        "train": np.arange(claim_offsets["train"], claim_offsets["train"] + len(frames[0])),
        "valid": np.arange(claim_offsets["valid"], claim_offsets["valid"] + len(frames[1])),
        "test": np.arange(claim_offsets["test"], claim_offsets["test"] + len(frames[2])),
    }
    return features, adjacency, idx, labels


def liar_metadata_gnn_baseline(task_name, splits, label_names, epochs=220, hidden_dim=128, patience=25):
    num_labels = len(label_names)
    features_np, adjacency_cpu, idx_np, labels_np = build_liar_metadata_graph(splits, num_labels)
    graph_device = device if torch.cuda.is_available() else torch.device("cpu")
    features = torch.tensor(features_np, dtype=torch.float32, device=graph_device)
    adjacency = adjacency_cpu.to(graph_device)
    train_idx = torch.tensor(idx_np["train"], dtype=torch.long, device=graph_device)
    valid_idx = torch.tensor(idx_np["valid"], dtype=torch.long, device=graph_device)
    test_idx = torch.tensor(idx_np["test"], dtype=torch.long, device=graph_device)
    y_train = torch.tensor(labels_np["train"], dtype=torch.long, device=graph_device)
    y_valid = torch.tensor(labels_np["valid"], dtype=torch.long, device=graph_device)

    model = SimpleGCN(features.shape[1], hidden_dim, num_labels, dropout=0.30).to(graph_device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=5e-4)
    weights = get_class_weights(labels_np["train"], num_labels).to(graph_device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

    best_state = None
    best_score = -1.0
    bad_epochs = 0
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(features, adjacency)
        loss = loss_fn(logits[train_idx], y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits = model(features, adjacency)
            valid_preds = logits[valid_idx].argmax(dim=1).detach().cpu().numpy()
        valid_score = f1_score(labels_np["valid"], valid_preds, average="macro", zero_division=0)
        if valid_score > best_score:
            best_score = valid_score
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if bad_epochs >= patience:
            break

    if best_state is not None:
        model.load_state_dict({key: value.to(graph_device) for key, value in best_state.items()})
    model.eval()
    with torch.no_grad():
        logits = model(features, adjacency)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()

    valid_probs = probs[idx_np["valid"]]
    test_probs = probs[idx_np["test"]]
    valid_summary = print_report(
        f"{task_name} valid - LIAR metadata GNN",
        labels_np["valid"],
        valid_probs.argmax(axis=1),
        label_names,
        task_name,
    )
    test_summary = print_report(
        f"{task_name} test - LIAR metadata GNN",
        labels_np["test"],
        test_probs.argmax(axis=1),
        label_names,
        task_name,
    )
    return {
        "valid_probs": valid_probs,
        "test_probs": test_probs,
        "valid_summary": valid_summary,
        "test_summary": test_summary,
        "test_preds": test_probs.argmax(axis=1),
    }


def tune_gnn_transformer_blend(task_name, y_valid, y_test, transformer_valid_probs, transformer_test_probs, gnn_valid_probs, gnn_test_probs, label_names):
    rows = []
    for alpha in np.linspace(0.0, 1.0, 51):
        valid_probs = alpha * transformer_valid_probs + (1.0 - alpha) * gnn_valid_probs
        valid_preds = valid_probs.argmax(axis=1)
        rows.append({"alpha_transformer": float(alpha), **summarize_predictions(y_valid, valid_preds, label_names, task_name)})
    table = pd.DataFrame(rows)
    best = table.sort_values(["accuracy", "macro_f1", "macro_recall"], ascending=False).iloc[0]
    test_probs = best["alpha_transformer"] * transformer_test_probs + (1.0 - best["alpha_transformer"]) * gnn_test_probs
    test_preds = test_probs.argmax(axis=1)
    summary = print_report(
        f"{task_name} test - Transformer + LIAR metadata GNN blend",
        y_test,
        test_preds,
        label_names,
        task_name,
    )
    print("Best GNN blend alpha on validation:", round(float(best["alpha_transformer"]), 4))
    return {"best": best, "table": table, "test_probs": test_probs, "test_preds": test_preds, "summary": summary}


In [11]:
# Binary ensemble evaluation and threshold tuning.
binary_ready_artifacts = get_ready_artifacts("binary", BINARY_MODELS)
binary_refit_ready_artifacts = get_ready_artifacts("binary_refit", BINARY_REFIT_MODELS)
binary_all_ready_artifacts = binary_ready_artifacts + binary_refit_ready_artifacts
binary_valid_probs, binary_test_probs = collect_probabilities(
    binary_all_ready_artifacts,
    binary_splits,
    num_labels=len(BINARY_LABEL_NAMES),
    max_length=160,
)

y_valid_binary = binary_splits["valid"]["labels"]
y_test_binary = binary_splits["test"]["labels"]

binary_test_ensemble = evaluate_ensembles(
    binary_valid_probs,
    binary_test_probs,
    y_valid_binary,
    y_test_binary,
    BINARY_LABEL_NAMES,
    "binary",
    "test",
)

if binary_valid_probs:
    selected_valid_models = binary_test_ensemble.get("validation_selected_avg", {}).get("models", tuple(sorted(binary_valid_probs)))
    _, valid_selected_probs = subset_probability_vote(binary_valid_probs, selected_valid_models)
    _, test_selected_probs = subset_probability_vote(binary_test_probs, selected_valid_models)

    best_accuracy_threshold, threshold_table = tune_binary_threshold(
        valid_selected_probs[:, 0],
        y_valid_binary,
        optimize_for="accuracy",
    )
    print("\nBest validation threshold for binary accuracy")
    print(best_accuracy_threshold.round(4).to_string())

    threshold_preds_test = np.where(test_selected_probs[:, 0] >= best_accuracy_threshold["threshold"], 0, 1)
    threshold_test_summary = print_report(
        "binary test - Validation-selected ensemble + tuned accuracy threshold",
        y_test_binary,
        threshold_preds_test,
        BINARY_LABEL_NAMES,
        "binary",
    )

binary_stacked_results = stacked_meta_ensemble(
    "binary",
    binary_splits,
    binary_valid_probs,
    binary_test_probs,
    BINARY_LABEL_NAMES,
)

binary_gnn_results = liar_metadata_gnn_baseline("binary", binary_splits, BINARY_LABEL_NAMES)

if binary_valid_probs:
    _, binary_transformer_valid_avg = summed_probability_vote(binary_valid_probs)
    _, binary_transformer_test_avg = summed_probability_vote(binary_test_probs)
    binary_gnn_blend_results = tune_gnn_transformer_blend(
        "binary",
        y_valid_binary,
        y_test_binary,
        binary_transformer_valid_avg,
        binary_transformer_test_avg,
        binary_gnn_results["valid_probs"],
        binary_gnn_results["test_probs"],
        BINARY_LABEL_NAMES,
    )

binary_predictions_for_report = {}
for model_name, probs in sorted(binary_test_probs.items()):
    binary_predictions_for_report[f"Individual: {model_name}"] = probs.argmax(axis=1)
for key, label in [
    ("majority_vote", "Ensemble: majority vote"),
    ("highest_sum", "Ensemble: highest-sum / avg prob"),
    ("validation_weighted", "Ensemble: validation-weighted avg prob"),
    ("validation_selected_avg", "Ensemble: validation-selected avg prob"),
]:
    if key in binary_test_ensemble:
        binary_predictions_for_report[label] = binary_test_ensemble[key]["preds"]
if "threshold_preds_test" in globals():
    binary_predictions_for_report["Ensemble: validation-selected + tuned threshold"] = threshold_preds_test
if "binary_gnn_results" in globals():
    binary_predictions_for_report["GNN: LIAR metadata graph"] = binary_gnn_results["test_preds"]
if "binary_gnn_blend_results" in globals():
    binary_predictions_for_report["Ensemble: transformer + LIAR metadata GNN"] = binary_gnn_blend_results["test_preds"]
if isinstance(binary_stacked_results, dict):
    if "tfidf_summary" in binary_stacked_results:
        binary_predictions_for_report["Baseline: TF-IDF + metadata"] = binary_stacked_results.get("tfidf_test_preds", np.array([]))
    if "credibility_test_preds" in binary_stacked_results:
        binary_predictions_for_report["Baseline: LIAR credibility"] = binary_stacked_results["credibility_test_preds"]
    if "meta_test_preds" in binary_stacked_results:
        binary_predictions_for_report["Ensemble: stacked transformer + credibility meta"] = binary_stacked_results["meta_test_preds"]

# Fix baseline predictions for report if the summary-only branch above was created.
if "Baseline: TF-IDF + metadata" in binary_predictions_for_report and len(binary_predictions_for_report["Baseline: TF-IDF + metadata"]) == 0:
    binary_predictions_for_report.pop("Baseline: TF-IDF + metadata")


Predicting probabilities: distilbert-base-uncased [binary] (max_length=128, batch_size=32)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 4959.22it/s]


Predicting probabilities: bert-base-uncased [binary] (max_length=128, batch_size=32)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 5025.06it/s]


Predicting probabilities: roberta-base [binary] (max_length=128, batch_size=32)


...

Predicting probabilities: bert-base-uncased [binary_refit] (max_length=128, batch_size=32)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 4568.12it/s]


Predicting probabilities: roberta-base [binary_refit] (max_length=128, batch_size=32)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 4466.90it/s]



binary test - Majority vote
accuracy           0.7607
macro_recall       0.7542
weighted_recall    0.7607
macro_f1           0.7552
weighted_f1        0.7601
fake_recall        0.7050
real_recall        0.8033
              precision    recall  f1-score   support

        fake       0.73      0.71      0.72       556
        real       0.78      0.80      0.79       727

    accuracy                           0.76      1283
   macro avg       0.76      0.75      0.76      1283
weighted avg       0.76      0.76      0.76      1283


binary test - Highest-sum / averaged probabilities
accuracy           0.7607
macro_recall       0.7542
weighted_recall    0.7607
macro_f1           0.7552
weighted_f1        0.7601
fake_recall        0.7050
real_recall        0.8033
              precision    recall  f1-score   support

        fake       0.73      0.71      0.72       556
        real       0.78      0.80      0.79       727

    accuracy                           0.76      1283
   macro a

In [12]:
# Multiclass and hierarchical fine-tuning.
RUN_MULTICLASS_TRAINING = True
MULTICLASS_MODELS = DEFAULT_MODELS

# Lightweight subtype hierarchy is enabled with DistilBERT only for the unattended full run.
RUN_HIERARCHICAL_SUBTYPE_TRAINING = True
HIERARCHICAL_SUBTYPE_MODELS = HIERARCHICAL_MODELS

multiclass_artifacts = []
if RUN_MULTICLASS_TRAINING:
    for model_name in MULTICLASS_MODELS:
        artifact = train_or_load_transformer(
            model_name=model_name,
            splits=multiclass_splits,
            task_name="multiclass",
            label_names=MULTICLASS_LABEL_NAMES,
            force_retrain=FORCE_RETRAIN,
            use_class_weights=True,
        )
        if not artifact.get("skipped"):
            multiclass_artifacts.append(artifact)

fake_subtype_artifacts = []
real_subtype_artifacts = []
if RUN_HIERARCHICAL_SUBTYPE_TRAINING:
    for model_name in HIERARCHICAL_SUBTYPE_MODELS:
        artifact = train_or_load_transformer(
            model_name=model_name,
            splits=fake_subtype_splits,
            task_name="fake_subtype",
            label_names=FAKE_SUBTYPE_LABEL_NAMES,
            force_retrain=FORCE_RETRAIN,
            use_class_weights=True,
        )
        if not artifact.get("skipped"):
            fake_subtype_artifacts.append(artifact)

    for model_name in HIERARCHICAL_SUBTYPE_MODELS:
        artifact = train_or_load_transformer(
            model_name=model_name,
            splits=real_subtype_splits,
            task_name="real_subtype",
            label_names=REAL_SUBTYPE_LABEL_NAMES,
            force_retrain=FORCE_RETRAIN,
            use_class_weights=True,
        )
        if not artifact.get("skipped"):
            real_subtype_artifacts.append(artifact)

multiclass_artifacts, fake_subtype_artifacts, real_subtype_artifacts


Training distilbert-base-uncased for multiclass -> C:\Users\Asus\Downloads\Fake news detection\code\optimized_results_simple_liar_novel_v7\distilbert_base_uncased_multiclass


Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 5554.71it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOG

Epoch,Training Loss,Validation Loss,Accuracy,Macro Recall,Weighted Recall,Macro F1,Weighted F1
1,1.444395,1.394866,0.406542,0.422606,0.406542,0.400077,0.390814
2,1.317350,1.289868,0.468847,0.479180,0.468847,0.465134,0.464489


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.91it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Macro Recall,Weighted Recall,Macro F1,Weighted F1
1.317350,1.289868,2,0.468847,0.479180,0.468847,0.465134,0.464489


...

Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.96it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Macro Recall,Weighted Recall,Macro F1,Weighted F1
0.826135,0.843455,2,0.595808,0.573119,0.595808,0.574026,0.589294


Training Loss,Validation Loss,Epoch,Accuracy,Macro Recall,Weighted Recall,Macro F1,Weighted F1
0.826135,0.867873,2,0.568088,0.558812,0.568088,0.559424,0.564316



distilbert-base-uncased real_subtype validation metrics
eval_loss               0.8435
eval_accuracy           0.5958
eval_macro_recall       0.5731
eval_weighted_recall    0.5958
eval_macro_f1           0.5740
eval_weighted_f1        0.5893

distilbert-base-uncased real_subtype test metrics
eval_loss               0.8679
eval_accuracy           0.5681
eval_macro_recall       0.5588
eval_weighted_recall    0.5681
eval_macro_f1           0.5594
eval_weighted_f1        0.5643


([{'model_name': 'distilbert-base-uncased',
   'task_name': 'multiclass',
   'run_dir': WindowsPath('C:/Users/Asus/Downloads/Fake news detection/code/optimized_results_simple_liar_novel_v7/distilbert_base_uncased_multiclass'),
   'valid_metrics': {'eval_loss': 1.2898681163787842,
    'eval_accuracy': 0.4688473520249221,
    'eval_macro_recall': 0.47918008262680517,
    'eval_weighted_recall': 0.4688473520249221,
    'eval_macro_f1': 0.4651339200748284,
    'eval_weighted_f1': 0.4644888180609851},
   'test_metrics': {'eval_loss': 1.3194626569747925,
    'eval_accuracy': 0.4169914263445051,
    'eval_macro_recall': 0.44442711019793824,
    'eval_weighted_recall': 0.4169914263445051,
    'eval_macro_f1': 0.41939449098595855,
    'eval_weighted_f1': 0.4115745420616898}},
  {'model_name': 'bert-base-uncased',
   'task_name': 'multiclass',
   'run_dir': WindowsPath('C:/Users/Asus/Downloads/Fake news detection/code/optimized_results_simple_liar_novel_v7/bert_base_uncased_multiclass'),
   'val

In [13]:
# Multiclass ensemble evaluation.
multiclass_ready_artifacts = get_ready_artifacts("multiclass", MULTICLASS_MODELS)
multiclass_valid_probs, multiclass_test_probs = collect_probabilities(
    multiclass_ready_artifacts,
    multiclass_splits,
    num_labels=len(MULTICLASS_LABEL_NAMES),
    max_length=160,
)

y_valid_multi = multiclass_splits["valid"]["labels"]
y_test_multi = multiclass_splits["test"]["labels"]

multiclass_test_ensemble = evaluate_ensembles(
    multiclass_valid_probs,
    multiclass_test_probs,
    y_valid_multi,
    y_test_multi,
    MULTICLASS_LABEL_NAMES,
    "multiclass",
    "test",
)

multiclass_stacked_results = stacked_meta_ensemble(
    "multiclass",
    multiclass_splits,
    multiclass_valid_probs,
    multiclass_test_probs,
    MULTICLASS_LABEL_NAMES,
)


multiclass_gnn_results = liar_metadata_gnn_baseline("multiclass", multiclass_splits, MULTICLASS_LABEL_NAMES)

if multiclass_valid_probs:
    _, multiclass_transformer_valid_avg = summed_probability_vote(multiclass_valid_probs)
    _, multiclass_transformer_test_avg = summed_probability_vote(multiclass_test_probs)
    multiclass_gnn_blend_results = tune_gnn_transformer_blend(
        "multiclass",
        y_valid_multi,
        y_test_multi,
        multiclass_transformer_valid_avg,
        multiclass_transformer_test_avg,
        multiclass_gnn_results["valid_probs"],
        multiclass_gnn_results["test_probs"],
        MULTICLASS_LABEL_NAMES,
    )


def hierarchical_multiclass_from_binary(binary_prob_dict, multiclass_prob_dict, threshold=0.5):
    if not binary_prob_dict or not multiclass_prob_dict:
        return None
    binary_avg = average_probabilities(binary_prob_dict)
    multiclass_avg = average_probabilities(multiclass_prob_dict)
    preds = []
    for fake_prob, class_probs in zip(binary_avg[:, 0], multiclass_avg):
        if fake_prob >= threshold:
            candidates = np.array([0, 1, 2])
        else:
            candidates = np.array([3, 4, 5])
        preds.append(candidates[np.argmax(class_probs[candidates])])
    return np.asarray(preds)


def tune_hierarchical_threshold_binary_plus_multiclass(binary_valid_prob_dict, multiclass_valid_prob_dict, y_valid):
    rows = []
    for threshold in np.linspace(0.20, 0.80, 121):
        preds = hierarchical_multiclass_from_binary(binary_valid_prob_dict, multiclass_valid_prob_dict, threshold)
        rows.append({
            "threshold": float(threshold),
            "accuracy": accuracy_score(y_valid, preds),
            "macro_recall": recall_score(y_valid, preds, average="macro", zero_division=0),
            "macro_f1": f1_score(y_valid, preds, average="macro", zero_division=0),
        })
    table = pd.DataFrame(rows)
    best = table.sort_values(["macro_f1", "accuracy", "macro_recall"], ascending=False).iloc[0]
    return best, table


if "binary_valid_probs" in globals() and binary_valid_probs and multiclass_valid_probs:
    hierarchical_best_threshold, hierarchical_threshold_table = tune_hierarchical_threshold_binary_plus_multiclass(
        binary_valid_probs,
        multiclass_valid_probs,
        y_valid_multi,
    )
    print("\nBest hierarchical binary + six-class validation threshold")
    print(hierarchical_best_threshold.round(4).to_string())

    hierarchical_multiclass_test_preds = hierarchical_multiclass_from_binary(
        binary_test_probs,
        multiclass_test_probs,
        threshold=hierarchical_best_threshold["threshold"],
    )
    hierarchical_multiclass_summary = print_report(
        "multiclass test - Hierarchical binary + six-class ensemble",
        y_test_multi,
        hierarchical_multiclass_test_preds,
        MULTICLASS_LABEL_NAMES,
        "multiclass",
    )


# Specialized hierarchy: binary gate + separate fake-subtype and real-subtype transformer ensembles.
fake_subtype_ready_artifacts = get_ready_artifacts("fake_subtype", HIERARCHICAL_SUBTYPE_MODELS)
real_subtype_ready_artifacts = get_ready_artifacts("real_subtype", HIERARCHICAL_SUBTYPE_MODELS)

if fake_subtype_ready_artifacts and real_subtype_ready_artifacts and binary_valid_probs:
    fake_subtype_valid_all_probs = collect_probabilities_for_texts(
        fake_subtype_ready_artifacts,
        multiclass_splits["valid"]["texts"],
        num_labels=len(FAKE_SUBTYPE_LABEL_NAMES),
        split_name="multiclass-valid fake-subtype",
        max_length=160,
    )
    fake_subtype_test_all_probs = collect_probabilities_for_texts(
        fake_subtype_ready_artifacts,
        multiclass_splits["test"]["texts"],
        num_labels=len(FAKE_SUBTYPE_LABEL_NAMES),
        split_name="multiclass-test fake-subtype",
        max_length=160,
    )
    real_subtype_valid_all_probs = collect_probabilities_for_texts(
        real_subtype_ready_artifacts,
        multiclass_splits["valid"]["texts"],
        num_labels=len(REAL_SUBTYPE_LABEL_NAMES),
        split_name="multiclass-valid real-subtype",
        max_length=160,
    )
    real_subtype_test_all_probs = collect_probabilities_for_texts(
        real_subtype_ready_artifacts,
        multiclass_splits["test"]["texts"],
        num_labels=len(REAL_SUBTYPE_LABEL_NAMES),
        split_name="multiclass-test real-subtype",
        max_length=160,
    )

    def specialized_hierarchical_predictions(binary_prob_dict, fake_prob_dict, real_prob_dict, threshold=0.5):
        binary_avg = average_probabilities(binary_prob_dict)
        fake_avg = average_probabilities(fake_prob_dict)
        real_avg = average_probabilities(real_prob_dict)
        preds = []
        for fake_gate_prob, fake_probs, real_probs in zip(binary_avg[:, 0], fake_avg, real_avg):
            if fake_gate_prob >= threshold:
                preds.append(int(np.argmax(fake_probs)))
            else:
                preds.append(int(np.argmax(real_probs)) + 3)
        return np.asarray(preds)

    def tune_specialized_hierarchical_threshold(binary_prob_dict, fake_prob_dict, real_prob_dict, y_valid):
        rows = []
        for threshold in np.linspace(0.20, 0.80, 121):
            preds = specialized_hierarchical_predictions(binary_prob_dict, fake_prob_dict, real_prob_dict, threshold)
            rows.append({
                "threshold": float(threshold),
                "accuracy": accuracy_score(y_valid, preds),
                "macro_recall": recall_score(y_valid, preds, average="macro", zero_division=0),
                "macro_f1": f1_score(y_valid, preds, average="macro", zero_division=0),
            })
        table = pd.DataFrame(rows)
        best = table.sort_values(["macro_f1", "accuracy", "macro_recall"], ascending=False).iloc[0]
        return best, table

    specialized_hierarchical_best_threshold, specialized_hierarchical_threshold_table = tune_specialized_hierarchical_threshold(
        binary_valid_probs,
        fake_subtype_valid_all_probs,
        real_subtype_valid_all_probs,
        y_valid_multi,
    )
    print("\nBest specialized hierarchical validation threshold")
    print(specialized_hierarchical_best_threshold.round(4).to_string())

    specialized_hierarchical_test_preds = specialized_hierarchical_predictions(
        binary_test_probs,
        fake_subtype_test_all_probs,
        real_subtype_test_all_probs,
        threshold=specialized_hierarchical_best_threshold["threshold"],
    )
    specialized_hierarchical_summary = print_report(
        "multiclass test - Specialized hierarchical binary + subtype ensembles",
        y_test_multi,
        specialized_hierarchical_test_preds,
        MULTICLASS_LABEL_NAMES,
        "multiclass",
    )


multiclass_predictions_for_report = {}
for model_name, probs in sorted(multiclass_test_probs.items()):
    multiclass_predictions_for_report[f"Individual: {model_name}"] = probs.argmax(axis=1)
for key, label in [
    ("majority_vote", "Ensemble: majority vote"),
    ("highest_sum", "Ensemble: highest-sum / avg prob"),
    ("validation_weighted", "Ensemble: validation-weighted avg prob"),
    ("validation_selected_avg", "Ensemble: validation-selected avg prob"),
]:
    if key in multiclass_test_ensemble:
        multiclass_predictions_for_report[label] = multiclass_test_ensemble[key]["preds"]
if isinstance(multiclass_stacked_results, dict) and "tfidf_test_preds" in multiclass_stacked_results:
    multiclass_predictions_for_report["Baseline: TF-IDF + metadata"] = multiclass_stacked_results["tfidf_test_preds"]
if isinstance(multiclass_stacked_results, dict) and "credibility_test_preds" in multiclass_stacked_results:
    multiclass_predictions_for_report["Baseline: LIAR credibility"] = multiclass_stacked_results["credibility_test_preds"]
if isinstance(multiclass_stacked_results, dict) and "meta_test_preds" in multiclass_stacked_results:
    multiclass_predictions_for_report["Ensemble: stacked transformer + credibility meta"] = multiclass_stacked_results["meta_test_preds"]
if "multiclass_gnn_results" in globals():
    multiclass_predictions_for_report["GNN: LIAR metadata graph"] = multiclass_gnn_results["test_preds"]
if "multiclass_gnn_blend_results" in globals():
    multiclass_predictions_for_report["Ensemble: transformer + LIAR metadata GNN"] = multiclass_gnn_blend_results["test_preds"]
if "hierarchical_multiclass_test_preds" in globals():
    multiclass_predictions_for_report["Ensemble: hierarchical binary + six-class"] = hierarchical_multiclass_test_preds
if "specialized_hierarchical_test_preds" in globals():
    multiclass_predictions_for_report["Ensemble: specialized hierarchical subtype"] = specialized_hierarchical_test_preds


Predicting probabilities: distilbert-base-uncased [multiclass] (max_length=128, batch_size=32)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 5778.43it/s]


Predicting probabilities: bert-base-uncased [multiclass] (max_length=128, batch_size=32)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 4674.26it/s]


Predicting probabilities: roberta-base [multiclass] (max_length=128, batch_size=32)


...

Predicting multiclass-valid real-subtype probabilities: distilbert-base-uncased [real_subtype] (max_length=128, batch_size=32)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 4163.24it/s]


Predicting multiclass-test real-subtype probabilities: distilbert-base-uncased [real_subtype] (max_length=128, batch_size=32)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 4727.10it/s]



Best specialized hierarchical validation threshold
threshold       0.4450
accuracy        0.4836
macro_recall    0.4929
macro_f1        0.4776

multiclass test - Specialized hierarchical binary + subtype ensembles
accuracy           0.4451
macro_recall       0.4683
weighted_recall    0.4451
macro_f1           0.4440
weighted_f1        0.4416
              precision    recall  f1-score   support

  pants-fire       0.38      0.71      0.49        92
       false       0.47      0.43      0.45       250
 barely-true       0.40      0.38      0.39       214
   half-true       0.51      0.43      0.46       267
 mostly-true       0.43      0.53      0.48       249
        true       0.49      0.33      0.39       211

    accuracy                           0.45      1283
   macro avg       0.45      0.47      0.44      1283
weighted avg       0.45      0.45      0.44      1283



In [14]:
# Save a compact experiment summary for comparison across runs.
summary = {
    "binary_models": [
        {
            "model_name": a.get("model_name"),
            "run_dir": str(a.get("run_dir")),
            "valid_metrics": a.get("valid_metrics", {}),
            "test_metrics": a.get("test_metrics", {}),
        }
        for a in binary_artifacts
    ],
    "binary_threshold": best_accuracy_threshold.to_dict() if "best_accuracy_threshold" in globals() else None,
    "binary_threshold_test_summary": threshold_test_summary if "threshold_test_summary" in globals() else None,
    "binary_stacked": binary_stacked_results if "binary_stacked_results" in globals() else None,
    "multiclass_stacked": multiclass_stacked_results if "multiclass_stacked_results" in globals() else None,
    "multiclass_models": [
        {
            "model_name": a.get("model_name"),
            "run_dir": str(a.get("run_dir")),
            "valid_metrics": a.get("valid_metrics", {}),
            "test_metrics": a.get("test_metrics", {}),
        }
        for a in multiclass_artifacts
    ],
}

summary_path = SUMMARY_DIR / "optimized_liar_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Saved summary to {summary_path}")


Saved summary to C:\Users\Asus\Downloads\Fake news detection\code\optimized_summaries_simple_liar_novel_v7\optimized_liar_summary.json


In [15]:
# ===== Final structured tables for report =====
# Run this after the binary and multiclass ensemble evaluation cells.

def row_from_predictions(task_name, split_name, method, y_true, y_pred):
    row = {
        "task": task_name,
        "split": split_name,
        "method": method,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    if task_name == "binary":
        row["fake_recall"] = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
        row["real_recall"] = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    return row


def make_results_table(task_name, y_true, prediction_dict):
    rows = [
        row_from_predictions(task_name, "test", method, y_true, y_pred)
        for method, y_pred in prediction_dict.items()
    ]
    table = pd.DataFrame(rows)
    metric_cols = [c for c in ["accuracy", "macro_recall", "weighted_recall", "macro_f1", "weighted_f1", "fake_recall", "real_recall"] if c in table.columns]
    table[metric_cols] = table[metric_cols].astype(float).round(4)
    return table.sort_values(["accuracy", "macro_f1"], ascending=False).reset_index(drop=True)


binary_results_table = make_results_table(
    "binary",
    y_test_binary,
    binary_predictions_for_report if "binary_predictions_for_report" in globals() else {},
)

multiclass_results_table = make_results_table(
    "multiclass",
    y_test_multi,
    multiclass_predictions_for_report if "multiclass_predictions_for_report" in globals() else {},
)

SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
binary_table_path = SUMMARY_DIR / "binary_results_table.csv"
multiclass_table_path = SUMMARY_DIR / "multiclass_results_table.csv"
binary_results_table.to_csv(binary_table_path, index=False)
multiclass_results_table.to_csv(multiclass_table_path, index=False)

if "threshold_table" in globals():
    threshold_table.round(4).to_csv(SUMMARY_DIR / "binary_threshold_tuning_table.csv", index=False)
if "hierarchical_threshold_table" in globals():
    hierarchical_threshold_table.round(4).to_csv(SUMMARY_DIR / "hierarchical_threshold_tuning_table.csv", index=False)
if "specialized_hierarchical_threshold_table" in globals():
    specialized_hierarchical_threshold_table.round(4).to_csv(SUMMARY_DIR / "specialized_hierarchical_threshold_tuning_table.csv", index=False)

print("Binary report table")
display(binary_results_table)
if len(binary_results_table):
    best_binary_row = binary_results_table.iloc[0]
    best_binary_accuracy = float(best_binary_row["accuracy"])
    if best_binary_accuracy >= 0.80:
        print(f"\nBINARY TARGET REACHED: {best_binary_accuracy:.4f} accuracy using {best_binary_row['method']}")
    else:
        print(f"\nBinary best is {best_binary_accuracy:.4f}; gap to 0.8000 is {0.80 - best_binary_accuracy:.4f}.")
print(f"Saved: {binary_table_path}")

print("\nMulticlass report table")
display(multiclass_results_table)
print(f"Saved: {multiclass_table_path}")


Binary report table


,task,split,method,accuracy,macro_recall,weighted_recall,macro_f1,weighted_f1,fake_recall,real_recall
0,binary,test,Ensemble: validation-selected avg prob,0.7646,0.7568,0.7646,0.7584,0.7636,0.6978,0.8157
1,binary,test,Ensemble: validation-selected + tuned threshold,0.7646,0.7568,0.7646,0.7584,0.7636,0.6978,0.8157
2,binary,test,Ensemble: majority vote,0.7607,0.7542,0.7607,0.7552,0.7601,0.7050,0.8033
3,binary,test,Ensemble: highest-sum / avg prob,0.7607,0.7542,0.7607,0.7552,0.7601,0.7050,0.8033
4,binary,test,Ensemble: validation-weighted avg prob,0.7599,0.7535,0.7599,0.7545,0.7594,0.7050,0.8019
5,binary,test,Ensemble: transformer + LIAR metadata GNN,0.7576,0.7516,0.7576,0.7524,0.7572,0.7068,0.7964
6,binary,test,Individual: roberta-base [binary_refit],0.7537,0.7469,0.7537,0.7480,0.7531,0.6960,0.7978
7,binary,test,Individual: roberta-base [binary],0.7529,0.7488,0.7529,0.7486,0.7530,0.7176,0.7799
8,binary,test,Individual: distilbert-base-uncased [binary_re...,0.7521,0.7477,0.7521,0.7477,0.7521,0.7140,0.7813
9,binary,test,Individual: distilbert-base-uncased [binary],0.7514,0.7463,0.7514,0.7466,0.7512,0.7086,0.7840



Binary best is 0.7646; gap to 0.8000 is 0.0354.
Saved: C:\Users\Asus\Downloads\Fake news detection\code\optimized_summaries_simple_liar_novel_v7\binary_results_table.csv

Multiclass report table


,task,split,method,accuracy,macro_recall,weighted_recall,macro_f1,weighted_f1
0,multiclass,test,Ensemble: stacked transformer + credibility meta,0.4482,0.4724,0.4482,0.4546,0.4474
1,multiclass,test,Ensemble: transformer + LIAR metadata GNN,0.4482,0.4753,0.4482,0.4530,0.4446
2,multiclass,test,Ensemble: specialized hierarchical subtype,0.4451,0.4683,0.4451,0.4440,0.4416
3,multiclass,test,Individual: bert-base-uncased [multiclass],0.4443,0.4743,0.4443,0.4499,0.4398
4,multiclass,test,Ensemble: highest-sum / avg prob,0.4419,0.4719,0.4419,0.4461,0.4358
5,multiclass,test,Ensemble: hierarchical binary + six-class,0.4419,0.4724,0.4419,0.4447,0.4350
6,multiclass,test,Ensemble: majority vote,0.4412,0.4689,0.4412,0.4465,0.4364
7,multiclass,test,Ensemble: validation-weighted avg prob,0.4396,0.4699,0.4396,0.4438,0.4336
8,multiclass,test,GNN: LIAR metadata graph,0.4380,0.4636,0.4380,0.4427,0.4319
9,multiclass,test,Individual: distilbert-base-uncased [multiclass],0.4170,0.4444,0.4170,0.4194,0.4116


Saved: C:\Users\Asus\Downloads\Fake news detection\code\optimized_summaries_simple_liar_novel_v7\multiclass_results_table.csv


In [16]:
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)


def safe_filename(name):
    return (
        name.replace("/", "_")
            .replace(":", "")
            .replace(" ", "_")
            .replace("-", "_")
            .lower()
    )


def make_separate_per_class_tables(y_true, prediction_dict, label_names, task_name):
    all_tables = {}
    task_dir = SUMMARY_DIR / f"{task_name}_per_class_tables"
    task_dir.mkdir(parents=True, exist_ok=True)

    for method, y_pred in prediction_dict.items():
        report = classification_report(
            y_true,
            y_pred,
            target_names=label_names,
            output_dict=True,
            zero_division=0,
        )
        table = pd.DataFrame(report).T.round(4)
        table = table.loc[label_names + ["accuracy", "macro avg", "weighted avg"]]

        path = task_dir / f"{safe_filename(method)}.csv"
        table.to_csv(path)

        print("\n" + "=" * 80)
        print(f"{task_name.upper()} - {method}")
        print("=" * 80)
        display(table)
        print("Saved:", path)
        all_tables[method] = table

    long_rows = []
    for method, table in all_tables.items():
        for label in label_names:
            long_rows.append({
                "task": task_name,
                "method": method,
                "class": label,
                "precision": table.loc[label, "precision"],
                "recall": table.loc[label, "recall"],
                "f1_score": table.loc[label, "f1-score"],
                "support": table.loc[label, "support"],
            })
    long_table = pd.DataFrame(long_rows)
    long_path = SUMMARY_DIR / f"{task_name}_per_class_report_long.csv"
    long_table.to_csv(long_path, index=False)
    print("\nSaved long per-class table:", long_path)
    display(long_table)
    return all_tables, long_table


binary_separate_tables, binary_per_class_long = make_separate_per_class_tables(
    y_true=y_test_binary,
    prediction_dict=binary_predictions_for_report,
    label_names=BINARY_LABEL_NAMES,
    task_name="binary",
)

multiclass_separate_tables, multiclass_per_class_long = make_separate_per_class_tables(
    y_true=y_test_multi,
    prediction_dict=multiclass_predictions_for_report,
    label_names=MULTICLASS_LABEL_NAMES,
    task_name="multiclass",
)



BINARY - Individual: bert-base-uncased [binary]


,precision,recall,f1-score,support
fake,0.6945,0.7320,0.7128,556.0000
real,0.7862,0.7538,0.7697,727.0000
accuracy,0.7443,0.7443,0.7443,0.7443
macro avg,0.7404,0.7429,0.7412,1283.0000
weighted avg,0.7465,0.7443,0.7450,1283.0000


Saved: C:\Users\Asus\Downloads\Fake news detection\code\optimized_summaries_simple_liar_novel_v7\binary_per_class_tables\individual_bert_base_uncased_[binary].csv

BINARY - Individual: bert-base-uncased [binary_refit]


,precision,recall,f1-score,support
fake,0.7368,0.6547,0.6933,556.000
real,0.7567,0.8212,0.7876,727.000
accuracy,0.7490,0.7490,0.7490,0.749
macro avg,0.7467,0.7379,0.7405,1283.000
weighted avg,0.7481,0.7490,0.7467,1283.000


Saved: C:\Users\Asus\Downloads\Fake news detection\code\optimized_summaries_simple_liar_novel_v7\binary_per_class_tables\individual_bert_base_uncased_[binary_refit].csv

BINARY - Individual: distilbert-base-uncased [binary]


...

,precision,recall,f1-score,support
pants-fire,0.4231,0.7174,0.5323,92.0000
false,0.4837,0.4160,0.4473,250.0000
barely-true,0.3612,0.5654,0.4408,214.0000
half-true,0.5766,0.2959,0.3911,267.0000
mostly-true,0.4262,0.5221,0.4693,249.0000
true,0.4963,0.3175,0.3873,211.0000
accuracy,0.4419,0.4419,0.4419,0.4419
macro avg,0.4612,0.4724,0.4447,1283.0000
weighted avg,0.4692,0.4419,0.4350,1283.0000


Saved: C:\Users\Asus\Downloads\Fake news detection\code\optimized_summaries_simple_liar_novel_v7\multiclass_per_class_tables\ensemble_hierarchical_binary_+_six_class.csv

MULTICLASS - Ensemble: specialized hierarchical subtype


,precision,recall,f1-score,support
pants-fire,0.3757,0.7065,0.4906,92.0000
false,0.4696,0.4320,0.4500,250.0000
barely-true,0.4020,0.3832,0.3923,214.0000
half-true,0.5067,0.4270,0.4634,267.0000
mostly-true,0.4290,0.5341,0.4758,249.0000
true,0.4894,0.3270,0.3920,211.0000
accuracy,0.4451,0.4451,0.4451,0.4451
macro avg,0.4454,0.4683,0.4440,1283.0000
weighted avg,0.4547,0.4451,0.4416,1283.0000


Saved: C:\Users\Asus\Downloads\Fake news detection\code\optimized_summaries_simple_liar_novel_v7\multiclass_per_class_tables\ensemble_specialized_hierarchical_subtype.csv

Saved long per-class table: C:\Users\Asus\Downloads\Fake news detection\code\optimized_summaries_simple_liar_novel_v7\multiclass_per_class_report_long.csv


,task,method,class,precision,recall,f1_score,support
0,multiclass,Individual: bert-base-uncased [multiclass],pants-fire,0.4323,0.7283,0.5425,92.0
1,multiclass,Individual: bert-base-uncased [multiclass],false,0.5733,0.3440,0.4300,250.0
2,multiclass,Individual: bert-base-uncased [multiclass],barely-true,0.4409,0.4533,0.4470,214.0
3,multiclass,Individual: bert-base-uncased [multiclass],half-true,0.4802,0.3633,0.4136,267.0
4,multiclass,Individual: bert-base-uncased [multiclass],mostly-true,0.4119,0.5542,0.4726,249.0
...,...,...,...,...,...,...,...
79,multiclass,Ensemble: specialized hierarchical subtype,false,0.4696,0.4320,0.4500,250.0
80,multiclass,Ensemble: specialized hierarchical subtype,barely-true,0.4020,0.3832,0.3923,214.0
81,multiclass,Ensemble: specialized hierarchical subtype,half-true,0.5067,0.4270,0.4634,267.0
82,multiclass,Ensemble: specialized hierarchical subtype,mostly-true,0.4290,0.5341,0.4758,249.0
